In [ ]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("PostgresPySpark") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4") \
    .enableHiveSupport() \
    .getOrCreate()

In [ ]:
spark.sql("USE prod_db")

In [6]:
import os
print(os.getcwd())

/home/jovyan/work


In [ ]:
spark.sql("""
WITH lineitem_agg AS (
    SELECT 
        l_orderkey,
        SUM(l_extendedprice) AS total_price_without_discount
    FROM 
        lineitem
    GROUP BY 
        l_orderkey
)
SELECT 
    o.o_orderkey,
    o.o_totalprice, 
    l.total_price_without_discount - o.o_totalprice AS amount_lost_to_discount
FROM 
    orders o
JOIN 
    lineitem_agg l ON o.o_orderkey = l.l_orderkey
ORDER BY 
    o.o_orderkey
""").show();

In [ ]:
#all the CTE less that 5 awlys
spark.sql("""
    with orders_cte as(
        select 
        o_orderkey, o_orderstatus, o_custkey,
        CAST(o_orderdate as TIMESTAMP) as o_orderdate,
        o_comment
        from orders 
    ),

    lineitem_cte as(
        select
        l_orderkey, l_partkey, l_linenumber, l_quantity, l_discount,
        cast(l_receiptdate as timestamp) as l_orderdatel
        from lineitem
    ),

    customer_cte as(
        select
        c_custkey, c_name, c_nationkey, c_nationkey
        from customer
    ),
    supplier_cte as(
        select
        s_suppkey, s_name, s_nationkey
        from supplier
    )
select o.o_orderkey, l.l_suppkey from orders o
join lineitem l on o.o_orderkey = l.l_orderkey
group by o.o_orderkey, l.l_suppkey
""").show()

In [ ]:
spark.sql(""" select * from lineitem
limit 4 """).show()

In [ ]:
#Sellers (name) who have sold at least one of the top 10 selling parts (use CTE)

spark.sql("""
    with top_part as(
    select l_partkey, sum(l_quantity) as ttl_qty from lineitem
    group by l_partkey
    order by ttl_qty desc
    limit 10
)
select distinct s.s_name as seller_name from 
lineitem l 
join top_part tp on  l.l_partkey = tp.l_partkey
join supplier s on l.l_suppkey = s.s_suppkey
""").show()